
# Peg Solitaire con A*: explicación paso a paso

El presente cuaderon expone la solución del juego **Peg Solitaire** con el algoritmo de busqueda A*

---

## Objetivo del problema

El tablero es el clásico English Cross de 7×7.  
Las reglas principales son:

- una ficha puede saltar sobre otra ficha vecina,
- la ficha saltada se elimina,
- el movimiento solo es válido si la casilla de destino está vacía,
- la meta es dejar una sola ficha en el centro.


## 1. Importaciones

Se utilizan dos librerias:

- `heapq` para implementar la **frontera** como un **min-heap**.
- `time` para medir el tiempo de ejecución.


In [7]:

import heapq
import time



## 2. Representación del tablero

El tablero se modela como una matriz 7×7:

- `-1` = celda inválida
- `1` = celda con ficha
- `0` = celda vacía

La máscara `VALID_MASK` define la forma de cruz del tablero inglés.


In [8]:

VALID_MASK = [
    [-1, -1,  1,  1,  1, -1, -1],
    [-1, -1,  1,  1,  1, -1, -1],
    [ 1,  1,  1,  1,  1,  1,  1],
    [ 1,  1,  1,  1,  1,  1,  1],
    [ 1,  1,  1,  1,  1,  1,  1],
    [-1, -1,  1,  1,  1, -1, -1],
    [-1, -1,  1,  1,  1, -1, -1],
]

VALID_POSITIONS = [(r, c) for r in range(7) for c in range(7) if VALID_MASK[r][c] != -1]
DIRECTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def create_initial_board():
    board = [row[:] for row in VALID_MASK]
    board[3][3] = 0
    return board

def print_board(board):
    symbols = {-1: " ", 1: "●", 0: "○"}
    for row in board:
        print("  ".join(symbols[c] for c in row))
    print()

board = create_initial_board()
print_board(board)


      ●  ●  ●      
      ●  ●  ●      
●  ●  ●  ●  ●  ●  ●
●  ●  ●  ○  ●  ●  ●
●  ●  ●  ●  ●  ●  ●
      ●  ●  ●      
      ●  ●  ●      




## 3. ¿Por qué usar listas y también tuplas?

Aquí hay una decisión de diseño muy importante.

### Listas para modificar el tablero
El tablero se crea como **lista de listas** porque eso facilita cambiar valores al aplicar un movimiento.

### Tuplas para guardar estados
Cuando A\* quiere registrar estados visitados en un `set`, necesita objetos **hashables**.  
Las listas no lo son porque son mutables. Por eso se convierte el tablero a **tupla de tuplas**.

En otras palabras:

- `list[list[int]]` → útil para **trabajar** y **modificar**
- `tuple[tuple[int]]` → útil para **comparar**, **guardar** y **hashear**

Eso permite hacer cosas como:

```python
closed_set.add(board_to_tuple(board))
```

y luego verificar si un estado ya fue visitado.


In [9]:

def board_to_tuple(board):
    return tuple(tuple(row) for row in board)

board_list = create_initial_board()
board_tuple = board_to_tuple(board_list)

print("Tipo de board_list:", type(board_list))
print("Tipo de board_tuple:", type(board_tuple))
print()
print("Representación tuple del tablero:")
print(board_tuple)


Tipo de board_list: <class 'list'>
Tipo de board_tuple: <class 'tuple'>

Representación tuple del tablero:
((-1, -1, 1, 1, 1, -1, -1), (-1, -1, 1, 1, 1, -1, -1), (1, 1, 1, 1, 1, 1, 1), (1, 1, 1, 0, 1, 1, 1), (1, 1, 1, 1, 1, 1, 1), (-1, -1, 1, 1, 1, -1, -1), (-1, -1, 1, 1, 1, -1, -1))



## 4. Hash y conjuntos de visitados

Cuando se usa un `set` en Python, internamente se apoya en **hashes** para buscar rápido.  
Aquí no definimos manualmente una función hash: Python la calcula automáticamente para la tupla.

Eso hace posible preguntar:

```python
if current._board_tuple in closed_set:
```

La idea es simple:

- si el estado ya fue procesado, se ignora;
- si no, se agrega al conjunto de cerrados.

Así se evita expandir el mismo tablero muchas veces.


In [10]:

closed_set = set()
closed_set.add(board_to_tuple(board_list))

print("¿El tablero inicial ya está en closed_set?")
print(board_to_tuple(board_list) in closed_set)


¿El tablero inicial ya está en closed_set?
True



## 5. Generación de movimientos

Un movimiento válido en Peg Solitaire cumple:

1. la celda origen contiene una ficha,
2. la celda intermedia contiene otra ficha,
3. la celda destino está vacía,
4. el movimiento es horizontal o vertical.

La función `get_valid_moves` genera todos los movimientos legales desde un estado.


In [11]:

def get_valid_moves(board):
    moves = []
    for r, c in VALID_POSITIONS:
        if board[r][c] != 1:
            continue
        for dr, dc in DIRECTIONS:
            mr, mc = r + dr, c + dc
            tr, tc = r + 2 * dr, c + 2 * dc
            if 0 <= tr < 7 and 0 <= tc < 7:
                if board[mr][mc] == 1 and board[tr][tc] == 0:
                    moves.append((r, c, tr, tc))
    return moves

moves = get_valid_moves(create_initial_board())
moves[:10], len(moves)


([(1, 3, 3, 3), (3, 1, 3, 3), (3, 5, 3, 3), (5, 3, 3, 3)], 4)


## 6. Aplicar un movimiento

Cuando un movimiento es válido:

- el origen queda vacío,
- la ficha intermedia desaparece,
- el destino recibe la ficha.

Importante: se construye un **nuevo tablero** y no se modifica el anterior directamente.
Eso evita alterar accidentalmente otros nodos del árbol de búsqueda.


In [12]:

def apply_move(board, move):
    r, c, tr, tc = move
    new_board = [row[:] for row in board]
    mr, mc = (r + tr) // 2, (c + tc) // 2
    new_board[r][c] = 0
    new_board[mr][mc] = 0
    new_board[tr][tc] = 1
    return new_board

example_board = create_initial_board()
example_move = get_valid_moves(example_board)[0]

print("Movimiento de ejemplo:", example_move)
print("Antes:")
print_board(example_board)

new_example_board = apply_move(example_board, example_move)
print("Después:")
print_board(new_example_board)


Movimiento de ejemplo: (1, 3, 3, 3)
Antes:
      ●  ●  ●      
      ●  ●  ●      
●  ●  ●  ●  ●  ●  ●
●  ●  ●  ○  ●  ●  ●
●  ●  ●  ●  ●  ●  ●
      ●  ●  ●      
      ●  ●  ●      

Después:
      ●  ●  ●      
      ●  ○  ●      
●  ●  ●  ○  ●  ●  ●
●  ●  ●  ●  ●  ●  ●
●  ●  ●  ●  ●  ●  ●
      ●  ●  ●      
      ●  ●  ●      




## 7. Meta y heurística

La meta es muy específica:

- debe quedar exactamente **1 ficha**,
- y debe estar en la posición central `(3,3)`.

La heurística utilizada es:

\[
h(n) = \text{fichas restantes} - 1
\]

¿Por qué tiene sentido?

Porque cada movimiento elimina exactamente **una ficha**.  
Si quedan `k` fichas, como mínimo se necesitan `k - 1` movimientos para quedar con una sola.


In [13]:

def count_pegs(board):
    return sum(cell == 1 for row in board for cell in row)

def is_goal(board):
    return count_pegs(board) == 1 and board[3][3] == 1

def heuristic(board):
    return count_pegs(board) - 1

initial = create_initial_board()
print("Fichas iniciales:", count_pegs(initial))
print("Heurística inicial:", heuristic(initial))
print("¿Es meta?", is_goal(initial))


Fichas iniciales: 32
Heurística inicial: 31
¿Es meta? False



## 8. El nodo de A*

Cada nodo del árbol de búsqueda guarda:

- `board`: estado actual,
- `g`: costo acumulado desde el inicio,
- `h`: heurística,
- `f = g + h`,
- `parent`: nodo padre,
- `move`: movimiento que generó este estado,
- `_board_tuple`: versión hashable del tablero.

### ¿Por qué existe `__lt__`?
`heapq` necesita comparar nodos para mantener el heap ordenado.  
El método `__lt__` indica que un nodo es “menor” que otro si su `f` es menor.

Gracias a eso, cuando se usa `heappop`, el heap entrega el nodo con menor `f`.


In [14]:

class AStarNode:
    __slots__ = ['board', 'g', 'h', 'f', 'parent', 'move', '_board_tuple']

    def __init__(self, board, g, parent=None, move=None):
        self.board = board
        self.g = g
        self.h = heuristic(board)
        self.f = self.g + self.h
        self.parent = parent
        self.move = move
        self._board_tuple = board_to_tuple(board)

    def __lt__(self, other):
        return self.f < other.f

node_a = AStarNode(create_initial_board(), g=0)
node_b = AStarNode(apply_move(create_initial_board(), get_valid_moves(create_initial_board())[0]), g=1)

print("f(node_a):", node_a.f)
print("f(node_b):", node_b.f)
print("¿node_a < node_b?:", node_a < node_b)


f(node_a): 31
f(node_b): 31
¿node_a < node_b?: False



## 9. Heap: la frontera de búsqueda

A\* mantiene una **open list** o **frontera** con los nodos pendientes de explorar.  
Aquí se implementa con `heapq`, que en Python funciona como **min-heap**.

Eso significa que:

```python
current = heapq.heappop(open_list)
```

extrae el nodo con menor prioridad según `__lt__`, o sea, el de menor `f`.

### ¿Qué representa la frontera?
Son los estados que ya fueron generados, pero aún no han sido expandidos.


In [15]:

open_list = []
heapq.heappush(open_list, node_b)
heapq.heappush(open_list, node_a)

first = heapq.heappop(open_list)
print("Nodo extraído primero tiene f =", first.f)


Nodo extraído primero tiene f = 31



## 10. Reconstrucción del camino solución

Cuando A\* encuentra el objetivo, el nodo final tiene referencias a sus padres.  
Siguiendo esos padres hacia atrás se puede reconstruir todo el camino desde el inicio.


In [16]:

def reconstruct_path(node):
    path = []
    current = node
    while current is not None:
        path.append((current.move, current.board))
        current = current.parent
    path.reverse()
    return path



## 11. Implementación completa de A*

### Recorrido general de la búsqueda

El recorrido del algoritmo sigue esta lógica:

1. Se crea el nodo inicial.
2. Se inserta en la frontera (`open_list`).
3. Se extrae el nodo con menor `f`.
4. Si ya fue visitado, se descarta.
5. Si es meta, se reconstruye la ruta.
6. Si no es meta, se generan sucesores.
7. Los sucesores nuevos se agregan al heap.
8. El proceso continúa hasta encontrar solución o vaciar la frontera.

La estructura `closed_set` evita reprocesar estados repetidos.


In [17]:

def a_star_search(initial_board, progress_interval=50_000):
    start_time = time.time()

    start_node = AStarNode(initial_board, g=0)

    open_list = []
    heapq.heappush(open_list, start_node)

    closed_set = set()

    nodes_expanded = 0
    nodes_generated = 0

    while open_list:
        current = heapq.heappop(open_list)

        if current._board_tuple in closed_set:
            continue

        closed_set.add(current._board_tuple)
        nodes_expanded += 1

        if is_goal(current.board):
            elapsed = time.time() - start_time
            path = reconstruct_path(current)
            stats = {
                "tiempo_ejecucion_seg": round(elapsed, 4),
                "nodos_expandidos": nodes_expanded,
                "nodos_generados": nodes_generated,
                "movimientos": len(path) - 1,
                "estados_en_closed": len(closed_set),
            }
            return path, stats

        for move in get_valid_moves(current.board):
            new_board = apply_move(current.board, move)
            new_tuple = board_to_tuple(new_board)

            if new_tuple not in closed_set:
                child = AStarNode(new_board, g=current.g + 1, parent=current, move=move)
                nodes_generated += 1
                heapq.heappush(open_list, child)

        if progress_interval and nodes_expanded % progress_interval == 0:
            pegs = count_pegs(current.board)
            print(
                f"[Progreso] Expandidos: {nodes_expanded:,} | "
                f"Frontera: {len(open_list):,} | "
                f"Fichas nodo actual: {pegs}"
            )

    elapsed = time.time() - start_time
    stats = {
        "tiempo_ejecucion_seg": round(elapsed, 4),
        "nodos_expandidos": nodes_expanded,
        "nodos_generados": nodes_generated,
        "movimientos": -1,
        "estados_en_closed": len(closed_set),
    }
    return None, stats



## 12. Mostrar movimientos de forma legible

Esta función convierte una tupla de movimiento en una descripción más clara.


In [18]:

def describe_move(move):
    if move is None:
        return "Estado inicial"
    r, c, tr, tc = move
    mr, mc = (r + tr) // 2, (c + tc) // 2
    return f"({r},{c}) → ({tr},{tc})  [salta sobre ({mr},{mc})]"



## 13. Ejecución final de la búsqueda

En esta sección se ejecuta A\* sobre el tablero inicial y se muestran:

- las estadísticas,
- la cantidad de movimientos,
- y algunos pasos de la ruta solución.




In [19]:

board = create_initial_board()

path, stats = a_star_search(board, progress_interval=50_000)

print("Estadísticas:")
for k, v in stats.items():
    print(f"- {k}: {v}")

if path is None:
    print("\nNo se encontró solución.")
else:
    print(f"\n¡Solución encontrada en {stats['movimientos']} movimientos!")


Estadísticas:
- tiempo_ejecucion_seg: 0.1656
- nodos_expandidos: 2294
- nodos_generados: 3647
- movimientos: 31
- estados_en_closed: 2294

¡Solución encontrada en 31 movimientos!



## 14. Recorrido de la búsqueda y lectura de la solución

La búsqueda en A\* no recorre los estados “en línea recta”.  
Explora muchos tableros candidatos, pero siempre prioriza el nodo con menor `f`.

### Interpretación del recorrido

- `g` aumenta conforme se hacen más movimientos.
- `h` disminuye cuando quedan menos fichas.
- `f = g + h` combina ambas ideas.

Eso hace que la búsqueda no sea ciega: intenta equilibrar lo ya recorrido con lo que todavía parece faltar.

### Sobre el recorrido final
El camino solución que reconstruimos no muestra todos los nodos explorados, sino solo la **secuencia correcta desde el estado inicial hasta la meta**.


In [20]:

if path is not None:
    print("Primeros pasos de la solución:\n")
    for i, (move, state) in enumerate(path[:5]):
        print(f"Paso {i}: {describe_move(move)}")
        print_board(state)

    print("...")
    print(f"Total de pasos en la ruta: {len(path) - 1}")


Primeros pasos de la solución:

Paso 0: Estado inicial
      ●  ●  ●      
      ●  ●  ●      
●  ●  ●  ●  ●  ●  ●
●  ●  ●  ○  ●  ●  ●
●  ●  ●  ●  ●  ●  ●
      ●  ●  ●      
      ●  ●  ●      

Paso 1: (5,3) → (3,3)  [salta sobre (4,3)]
      ●  ●  ●      
      ●  ●  ●      
●  ●  ●  ●  ●  ●  ●
●  ●  ●  ●  ●  ●  ●
●  ●  ●  ○  ●  ●  ●
      ●  ○  ●      
      ●  ●  ●      

Paso 2: (4,5) → (4,3)  [salta sobre (4,4)]
      ●  ●  ●      
      ●  ●  ●      
●  ●  ●  ●  ●  ●  ●
●  ●  ●  ●  ●  ●  ●
●  ●  ●  ●  ○  ○  ●
      ●  ○  ●      
      ●  ●  ●      

Paso 3: (6,4) → (4,4)  [salta sobre (5,4)]
      ●  ●  ●      
      ●  ●  ●      
●  ●  ●  ●  ●  ●  ●
●  ●  ●  ●  ●  ●  ●
●  ●  ●  ●  ●  ○  ●
      ●  ○  ○      
      ●  ●  ○      

Paso 4: (6,2) → (6,4)  [salta sobre (6,3)]
      ●  ●  ●      
      ●  ●  ●      
●  ●  ●  ●  ●  ●  ●
●  ●  ●  ●  ●  ●  ●
●  ●  ●  ●  ●  ○  ●
      ●  ○  ○      
      ○  ○  ●      

...
Total de pasos en la ruta: 31


## 15. Desafios encontrados

- Se presentaron dificultades para definir la heuristica específica que se deseaba implenetar, sin esta, todas las ejecuciones no encontraban solucion o la encontraban después de tiempos excesivos de ejecución
- Se opto por añadir tuplas a la estructura general de resolucion dado a que comparar estados del tablero en forma de listas era extremadamente tardado y confirmar si un estado se encuentraba en una lista de estados aumentaba el tiempo de ejecución de forma exponencial según que tantas iteraciones se hayan implementado para resolver el algoritmo
- Se encontraron dificultades para presentar todo el recorrido del algoritmo para la resolución, por lo que se decidio añadir referenicias a movimientos y a padres de los nodos al nodo principal y crear una fucnión que se encargará de manejar este recorrido.

